In [ ]:
%sql
/* ===================================================================================
   DIAGNOSTIC: Full Comparison of Master AUs vs. CC Mapping
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(`US OR Canada`) AS Raw_Country_Value
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      -- Temporarily commenting out the strict filter so you can see the raw data
      -- AND UPPER(TRIM(`US OR Canada`)) = 'CANADA'
),

CC_Mapped_AUs AS (
    SELECT DISTINCT TRIM(CAST(AU_ID AS STRING)) AS Mapped_AU_ID
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
)

SELECT 
    COALESCE(m.BusinessID, c.Mapped_AU_ID) AS AU_ID,
    m.AU_Name,
    m.Raw_Country_Value,
    
    CASE 
        WHEN m.BusinessID IS NOT NULL AND c.Mapped_AU_ID IS NOT NULL THEN 'In BOTH Tables'
        WHEN m.BusinessID IS NOT NULL AND c.Mapped_AU_ID IS NULL THEN 'Master List ONLY (Missing from Mapping)'
        WHEN m.BusinessID IS NULL AND c.Mapped_AU_ID IS NOT NULL THEN 'CC Mapping ONLY (Missing from Master)'
    END AS Data_Source

FROM Master_AUs m
FULL OUTER JOIN CC_Mapped_AUs c
    ON m.BusinessID = c.Mapped_AU_ID
ORDER BY AU_ID;